# 08 — Report Analytics and Decision Support

Sprint 7 report-level analytics combining historical usage, forecast outlook,
model health, engagement context, metadata context, diagnostics, and segmentation
into the canonical `mart_report_analytics.csv`.

**All outputs are pre-computed and persisted. This notebook loads, explores, and explains them.**

## 1. Business Objective

The report analytics layer answers:
- **What has happened to report usage?** (historical performance, trends, inactivity)
- **What is forecast to happen?** (28-day and 7-day outlook, direction, low-usage risk)
- **How trustworthy is the forecast?** (model diagnostic status, evidence maturity, uncertainty)
- **How are users engaging?** (breadth, repeat, cohorts, frequency, concentration)
- **What lifecycle and metadata context is available?** (ownership, criticality, cadence, activation)
- **What deterministic risks and review actions should be considered?** (diagnostics, segments, priority)

### Important distinctions

| Concept | Is NOT the same as |
|---|---|
| Usage volume | Engagement quality |
| Forecast outlook | Model health |
| Low usage | Low business value |
| Concentration | Misuse |
| Recommended actions | Automated decisions |

All recommended actions in this layer are **review triggers for human decision-makers**, not automated interventions.
No pipeline in Sprint 7 recommends automatic report retirement, deletion, model retraining, or user restriction.

## 2. Architecture

```text
Historical report usage  (outputs/metrics/report_features.csv)
          +
Forecast outlook         (outputs/analytics/report_forecast_outlook.csv)
          +
Model-health evidence    (outputs/analytics/report_model_health_context.csv)
          +
User engagement          (outputs/analytics/report_engagement_context.csv)
          +
Report metadata          (outputs/analytics/report_metadata_context.csv)
                 ↓
Deterministic diagnostics (outputs/analytics/report_diagnostics.csv)
                 ↓
Dimensional segments      (outputs/analytics/report_segments.csv)
                 ↓
mart_report_analytics     (outputs/analytics/mart_report_analytics.csv)
```

### Source files and join keys

| Source | File | Grain | Key |
|---|---|---|---|
| Report features | `outputs/metrics/report_features.csv` | report_id | report_id |
| Forecast outlook | `outputs/analytics/report_forecast_outlook.csv` | report_id | report_id |
| Model health | `outputs/analytics/report_model_health_context.csv` | report_id | report_id |
| Engagement context | `outputs/analytics/report_engagement_context.csv` | report_id | report_id |
| Metadata context | `outputs/analytics/report_metadata_context.csv` | report_id | report_id |
| Diagnostics | `outputs/analytics/report_diagnostics.csv` | report_id | report_id |
| Segments | `outputs/analytics/report_segments.csv` | report_id | report_id |
| **Mart** | `outputs/analytics/mart_report_analytics.csv` | report_id | report_id |

All sources share `analytics_as_of_date` derived from `max(usage_date)` in the canonical mart — never `date.today()`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Repository-relative paths
ANALYTICS_DIR = Path("outputs/analytics")
METRICS_DIR = Path("outputs/metrics")

# Load all Sprint 7 outputs
features_df = pd.read_csv(METRICS_DIR / "report_features.csv")
forecast_df = pd.read_csv(ANALYTICS_DIR / "report_forecast_outlook.csv")
model_health_df = pd.read_csv(ANALYTICS_DIR / "report_model_health_context.csv")
engagement_df = pd.read_csv(ANALYTICS_DIR / "report_engagement_context.csv")
metadata_df = pd.read_csv(ANALYTICS_DIR / "report_metadata_context.csv")
diagnostics_df = pd.read_csv(ANALYTICS_DIR / "report_diagnostics.csv")
segments_df = pd.read_csv(ANALYTICS_DIR / "report_segments.csv")
mart_df = pd.read_csv(ANALYTICS_DIR / "mart_report_analytics.csv")

print(f"Reports in mart: {len(mart_df)}")
print(f"Analytics as-of: {mart_df['analytics_as_of_date'].iloc[0] if 'analytics_as_of_date' in mart_df.columns else 'N/A'}")

## 3. Historical Usage

Derived from `report_features.csv`. Each report has a complete daily spine
zero-filled for missing dates. `analytics_as_of_date` is derived from `max(usage_date)`
in the source mart — never `date.today()`.

**Zero vs. missing evidence:**
- Zero views on a given day = the report was available but had no activity
- `null` metrics = evidence was insufficient to calculate (e.g., fewer than 14 days of history for anomaly detection)

**Evidence-aware inactivity:** `inactivity_risk` is only raised when the report has a
sufficient observable usage window — it is not raised purely from a null history.

In [ ]:
usage_cols = [c for c in [
    "report_name", "historical_usage_status", "adoption_maturity_status",
    "recent_28d_views", "previous_28d_views", "usage_change_28d_pct",
    "active_usage_days_28d", "days_since_last_use", "report_age_days",
] if c in features_df.columns]

print("Historical usage summary:")
display(features_df[usage_cols].head(10))

fig, ax = plt.subplots(figsize=(10, 4))
if "historical_usage_status" in features_df.columns:
    status_counts = features_df["historical_usage_status"].value_counts().sort_index()
    status_counts.plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title("Historical Usage Status Distribution", fontsize=13)
    ax.set_xlabel("Number of reports")
    for i, v in enumerate(status_counts):
        ax.text(v + 0.1, i, str(v), va="center", fontsize=9)
plt.tight_layout()
plt.show()
print(f"Total reports with usage history: {len(features_df)}")

## 4. Forecast Outlook

Derived from `report_forecast_outlook.csv`. The forecast as-of date is derived as
`min(forecast_date) - 1 day` — aligned with the historical `analytics_as_of_date`.

**Key distinctions:**
- **Forecast direction** (growth/stable/decline) reflects the 28-day outlook
- **Within-horizon trend** (rising/flat/falling) describes how the horizon unfolds
- **Summed interval bounds** are conservative — they do not represent the actual range of the sum
- **Uncertainty** measures relative CI width; high uncertainty limits actionable interpretation

A declining forecast does not imply poor model health — those are independent signals.

In [ ]:
forecast_cols = [c for c in [
    "report_name", "forecast_outlook_status", "forecast_direction_28d",
    "forecast_uncertainty_status", "forecast_total_28d",
    "recent_actual_total_28d", "forecast_change_vs_actual_28d_pct",
    "forecast_trend_status", "recommended_forecast_review_action",
] if c in forecast_df.columns]

print("Forecast outlook summary:")
display(forecast_df[forecast_cols].head(10))

if "recent_actual_total_28d" in forecast_df.columns and "forecast_total_28d" in forecast_df.columns:
    fig, ax = plt.subplots(figsize=(11, 4))
    x = range(len(forecast_df))
    rnames = forecast_df["report_name"].tolist() if "report_name" in forecast_df.columns else [str(i) for i in x]
    ax.bar(x, forecast_df["recent_actual_total_28d"].fillna(0), label="Actual 28d", alpha=0.7, color="steelblue")
    ax.bar(x, forecast_df["forecast_total_28d"].fillna(0), label="Forecast 28d", alpha=0.6, color="orange", width=0.5)
    ax.set_xticks(list(x))
    ax.set_xticklabels(rnames, rotation=75, ha="right", fontsize=7)
    ax.set_title("Actual vs Forecast 28-Day Views by Report")
    ax.legend()
    plt.tight_layout()
    plt.show()

if "forecast_uncertainty_status" in forecast_df.columns:
    unc_counts = forecast_df["forecast_uncertainty_status"].value_counts()
    print("\nForecast uncertainty distribution:")
    print(unc_counts.to_string())

## 5. Model Health

Derived from `report_model_health_context.csv`. Model health reflects whether the
forecasting model's backtest and production evidence support reliable predictions.

**Examples to understand:**
- **Declining outlook + healthy model** → decline signal can be trusted
- **Declining outlook + poor model** → decline signal should be treated with caution
- **Stable outlook + insufficient evidence** → insufficient backtests; cannot confirm stability
- **Immature production evidence** → model has been in production fewer than 28 days

**Important:** Poor model health does not mean poor report performance. Model health
affects how much we can trust the forecast, not whether the report is useful.

In [ ]:
model_cols = [c for c in [
    "report_name", "model_diagnostic_status", "primary_model_issue",
    "production_evidence_maturity", "forecast_interpretation_status",
    "recommended_model_action", "model_review_required",
] if c in model_health_df.columns]

print("Model health summary:")
display(model_health_df[model_cols].head(10))

if "model_diagnostic_status" in model_health_df.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    mh_counts = model_health_df["model_diagnostic_status"].value_counts()
    colors = ["#2ecc71" if s in ("good", "healthy") else "#e74c3c" if s == "poor" else "#f39c12"
              for s in mh_counts.index]
    mh_counts.plot(kind="bar", ax=ax, color=colors)
    ax.set_title("Model Health Status Distribution")
    ax.set_xlabel("")
    ax.set_ylabel("Reports")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

## 6. Engagement

Derived from `report_engagement_context.csv`. Engagement measures *how users interact*
with a report — distinct from raw view counts.

**Key distinctions:**
- **Views vs users:** 100 views from 1 user differs from 100 views from 50 users
- **Returning users vs repeat views:** a user returning on separate days vs viewing multiple pages in one session
- **Healthy niche engagement vs concentrated dependency:**
  - Niche: small but stable and loyal user base — potentially healthy
  - Dependency: high concentration where the report's use is fragile if that user leaves
- **Privacy-limited evidence:** when fewer than 5 unique users are active in a window,
  distribution metrics are suppressed (`null`) — not treated as zero

In [ ]:
eng_cols = [c for c in [
    "report_name", "overall_engagement_status", "engagement_evidence_status",
    "unique_users_28d", "returning_user_share_28d", "lapse_rate_28d",
    "dependency_status", "privacy_suppression_status",
    "breadth_status", "frequency_direction",
] if c in engagement_df.columns]

print("Engagement summary:")
display(engagement_df[eng_cols].head(10))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
if "overall_engagement_status" in engagement_df.columns:
    eng_counts = engagement_df["overall_engagement_status"].value_counts()
    eng_counts.plot(kind="barh", ax=axes[0], color="teal")
    axes[0].set_title("Engagement Status Distribution")
    axes[0].set_xlabel("Reports")

if "unique_users_28d" in engagement_df.columns and "top_1_user_view_share_28d" in engagement_df.columns:
    if "privacy_suppression_status" in engagement_df.columns:
        unsup = engagement_df[
            ~engagement_df["privacy_suppression_status"].str.contains("suppressed", case=False, na=False)
        ]
    else:
        unsup = engagement_df
    if len(unsup) > 0:
        axes[1].scatter(unsup["unique_users_28d"], unsup["top_1_user_view_share_28d"], alpha=0.7, color="coral")
        axes[1].set_xlabel("Unique Users (28d)")
        axes[1].set_ylabel("Top-1 User View Share (28d)")
        axes[1].set_title("Concentration vs Active Users\n(unsuppressed only)")
    else:
        axes[1].text(0.5, 0.5, "All metrics suppressed", ha="center", va="center")
        axes[1].set_title("Concentration vs Active Users")
plt.tight_layout()
plt.show()

## 7. Metadata and Lifecycle Context

Derived from `report_metadata_context.csv`. Only **explicit** metadata fields from
`dim_report.csv` are used — business context is never inferred from usage patterns.

**Explicitly not inferred:**
- Cadence is not guessed from daily/weekly access patterns
- Criticality is not assumed from view volume
- Business value is not inferred from user count or concentration

A missing criticality field means "unknown" — it does not mean "low criticality".
The `metadata_completeness_score` (0–1) reflects what fraction of key fields are populated.

In [ ]:
meta_cols = [c for c in [
    "report_name", "metadata_interpretation_status", "metadata_completeness_score",
    "report_activation_date", "report_age_days", "adoption_maturity_status",
    "report_owner_team", "report_category", "expected_usage_cadence",
    "criticality_level", "certification_status", "deprecation_status",
] if c in metadata_df.columns]

print("Metadata summary:")
display(metadata_df[meta_cols].head(10))

if "metadata_completeness_score" in metadata_df.columns:
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.hist(metadata_df["metadata_completeness_score"].dropna(), bins=10, color="mediumpurple", edgecolor="white")
    ax.set_title("Metadata Completeness Score Distribution")
    ax.set_xlabel("Score (0 = no metadata, 1 = complete)")
    ax.set_ylabel("Reports")
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    plt.tight_layout()
    plt.show()
    print(f"Mean completeness: {metadata_df['metadata_completeness_score'].mean():.2f}")
    print(f"Reports with complete metadata: {(metadata_df['metadata_completeness_score'] == 1.0).sum()}")

## 8. Diagnostics

Derived from `report_diagnostics.csv`. Diagnostics are fully deterministic —
no randomness, no LLM calls. Each risk flag is evidence-gated: a risk is only
raised when supporting evidence is available.

**Risk categories:**
| Category | Raises risk when |
|---|---|
| historical_usage | Supported decline, inactivity, volatility, anomaly |
| forecast_outlook | Supported decline, low-usage, inactivity, high uncertainty |
| model_health | Poor/watch diagnostic status, individual model issues |
| engagement | Active-user decline, low repeat, elevated lapse, declining frequency |
| dependency | Unsuppressed high or increasing concentration |
| lifecycle | Immature report, deprecation context |
| metadata | Missing ownership, cadence, criticality |
| data_quality | No valid usage data or calculation failure |

**Deterministic precedence:** no_valid_data → prolonged_inactivity → severe_historical_decline
→ expected_inactivity → severe_model_health_issue → elevated_lapse → active_user_decline
→ concentrated_dependency → high_forecast_uncertainty → declining_frequency
→ low_repeat_engagement → metadata_limitation → newly_launched_or_immature → none

In [ ]:
diag_summary_cols = [c for c in [
    "report_name", "primary_diagnostic", "primary_diagnostic_category",
    "overall_diagnostic_severity", "recommended_diagnostic_action",
    "diagnostic_review_required", "diagnostic_issue_count", "diagnostic_warning_count",
] if c in diagnostics_df.columns]

print("Diagnostics summary:")
display(diagnostics_df[diag_summary_cols].head(10))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

if "primary_diagnostic_category" in diagnostics_df.columns:
    cat_counts = diagnostics_df["primary_diagnostic_category"].value_counts()
    cat_counts.plot(kind="barh", ax=axes[0], color="salmon")
    axes[0].set_title("Primary Diagnostic Category")
    axes[0].set_xlabel("Reports")

if "overall_diagnostic_severity" in diagnostics_df.columns:
    sev_order = ["poor", "warning", "informational", "none", "insufficient_evidence"]
    sev_counts = diagnostics_df["overall_diagnostic_severity"].value_counts()
    sev_counts = sev_counts.reindex([s for s in sev_order if s in sev_counts.index], fill_value=0)
    color_map = {"poor": "#e74c3c", "warning": "#f39c12", "informational": "#3498db",
                 "none": "#2ecc71", "insufficient_evidence": "#95a5a6"}
    sev_counts.plot(kind="bar", ax=axes[1],
                    color=[color_map.get(s, "gray") for s in sev_counts.index])
    axes[1].set_title("Diagnostic Severity")
    axes[1].set_xlabel("")
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha="right")

if "diagnostic_review_required" in diagnostics_df.columns:
    review_counts = diagnostics_df["diagnostic_review_required"].value_counts()
    axes[2].pie(review_counts, labels=review_counts.index, autopct="%1.0f%%",
                colors=["#e74c3c", "#2ecc71"])
    axes[2].set_title("Review Required")

plt.tight_layout()
plt.show()

## 9. Segmentation

Derived from `report_segments.csv`. Each dimensional segment is independent.
The primary segment uses deterministic 15-step precedence.

**Why the old `niche` segment was removed:**
- "Niche" previously conflated two unrelated concepts: small user base AND high concentration
- These are different signals requiring different responses:
  - **Healthy niche engagement** (`niche_healthy_engagement`): small, loyal, stable user base — potentially healthy
  - **Concentrated dependency** (`concentrated_dependency`): at-risk pattern where one or few users dominate access
- Conflating them obscured the actual risk profile

The new segmentation keeps them in separate dimensions: `engagement_segment` vs `dependency_segment`.

In [ ]:
seg_cols = [c for c in [
    "report_name", "usage_segment", "engagement_segment", "forecast_segment",
    "model_health_segment", "dependency_segment", "lifecycle_segment",
    "metadata_segment", "primary_report_segment", "segment_evidence_status",
] if c in segments_df.columns]

print("Segment summary:")
display(segments_df[seg_cols].head(10))

if "primary_report_segment" in segments_df.columns:
    fig, ax = plt.subplots(figsize=(10, 4))
    seg_counts = segments_df["primary_report_segment"].value_counts().sort_values()
    seg_counts.plot(kind="barh", ax=ax, color="darkcyan")
    ax.set_title("Primary Report Segment Distribution")
    ax.set_xlabel("Reports")
    for i, v in enumerate(seg_counts):
        ax.text(v + 0.05, i, str(v), va="center", fontsize=9)
    plt.tight_layout()
    plt.show()

## 10. Canonical Report Analytics Mart

`mart_report_analytics.csv` is the single source of truth combining all Sprint 7 layers.
It is the input for Sprint 8 (GenAI narrative generation) and Sprint 9 (Streamlit dashboard).

Sprint 7 itself does not modify GenAI or Streamlit.

In [ ]:
# Compact view — key decision fields only
mart_display_cols = [c for c in [
    "report_name", "historical_usage_status", "forecast_outlook_status",
    "model_diagnostic_status", "overall_engagement_status", "criticality_level",
    "primary_diagnostic", "primary_report_segment", "overall_report_status",
    "recommended_report_action", "overall_evidence_status", "overall_review_priority",
] if c in mart_df.columns]

print(f"Mart: {len(mart_df)} reports x {len(mart_df.columns)} columns")
display(mart_df[mart_display_cols])

if "overall_report_status" in mart_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    status_counts = mart_df["overall_report_status"].value_counts().sort_values()
    status_counts.plot(kind="barh", ax=axes[0], color="steelblue")
    axes[0].set_title("Overall Report Status Distribution")
    axes[0].set_xlabel("Reports")

    if "overall_review_priority" in mart_df.columns:
        pri_order = ["high", "medium", "low", "insufficient_evidence"]
        pri_counts = mart_df["overall_review_priority"].value_counts()
        pri_counts = pri_counts.reindex([p for p in pri_order if p in pri_counts.index], fill_value=0)
        colors_map = {"high": "#e74c3c", "medium": "#f39c12", "low": "#2ecc71",
                      "insufficient_evidence": "#95a5a6"}
        pri_counts.plot(kind="bar", ax=axes[1],
                        color=[colors_map.get(p, "gray") for p in pri_counts.index])
        axes[1].set_title("Review Priority Distribution")
        axes[1].set_xlabel("")
        plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

## 11. Representative Case Studies

Cases are selected deterministically from persisted outputs — one report per category.
Categories unavailable in the current dataset are skipped gracefully.

In [ ]:
def _first_match(df, col, values):
    """Return the first row where col is in values, or None."""
    if col not in df.columns:
        return None
    match = df[df[col].isin(values if isinstance(values, (list, set)) else [values])]
    return match.iloc[0] if len(match) > 0 else None

case_definitions = {
    "Healthy Broad Adoption": ("primary_report_segment", ["healthy_broad_adoption"]),
    "Healthy Niche Adoption": ("primary_report_segment", ["healthy_niche_adoption"]),
    "Growing Report": ("primary_report_segment", ["growing_report"]),
    "Declining Report": ("primary_report_segment", ["declining_report"]),
    "Inactive Report": ("primary_report_segment", ["inactive_report"]),
    "Elevated Lapse": ("primary_report_segment", ["elevated_lapse"]),
    "Low Repeat Usage": ("primary_report_segment", ["low_repeat_usage"]),
    "Concentrated Dependency": ("primary_report_segment", ["concentrated_dependency"]),
    "Uncertain Forecast": ("primary_report_segment", ["uncertain_forecast"]),
    "Model Review Needed": ("primary_report_segment", ["model_review_needed"]),
    "Newly Launched": ("primary_report_segment", ["newly_launched"]),
    "Planned Deprecation": ("primary_report_segment", ["planned_deprecation"]),
    "Mixed Signals": ("primary_report_segment", ["mixed_signals"]),
    "Insufficient Evidence": ("primary_report_segment", ["insufficient_evidence"]),
    "Data Quality Issue": ("primary_report_segment", ["data_quality_issue"]),
}

available_cases = []
skipped_cases = []

for label, (col, values) in case_definitions.items():
    row = _first_match(segments_df, col, values)
    if row is not None:
        available_cases.append((label, row["report_id"]))
    else:
        skipped_cases.append(label)

print(f"Available cases: {len(available_cases)}")
print(f"Skipped (not in dataset): {skipped_cases}")

detail_cols = [c for c in [
    "report_name", "overall_report_status", "primary_diagnostic",
    "primary_report_segment", "recommended_report_action",
    "overall_evidence_status", "overall_review_priority",
] if c in mart_df.columns]

for label, rid in available_cases:
    case_row = mart_df[mart_df["report_id"] == rid]
    if len(case_row) == 0:
        continue
    print(f"\n{'='*60}")
    print(f"Case: {label}  (report_id: {rid})")
    print(f"{'='*60}")
    display(case_row[detail_cols])

## 12. Evidence and Privacy Limitations

### Missing evidence
- Missing source files → `null` fields for that source (the report is kept in the mart)
- Temporal mismatches → flagged in alignment status fields
- Insufficient observable windows → metrics null, not zero
- Immature production evidence → model health limited

### Privacy suppression
When fewer than 5 unique users are active in a given window:
- Distribution metrics (HHI, concentration shares) are suppressed → `null`
- `privacy_suppression_status` and `privacy_suppressed_field_count` record what was suppressed
- Suppressed values are **never treated as zero**
- Concentration risk is **never raised from suppressed metrics**

### What missing values mean

| Field is null | Correct interpretation |
|---|---|
| `top_1_user_view_share_28d` | Privacy suppressed (< 5 users) — not "no concentration" |
| `lapse_rate_28d` | Insufficient cohort history — not "no lapse" |
| `forecast_uncertainty_status` | Forecast unavailable — not "low uncertainty" |
| `criticality_level` is "unknown" | Not recorded — not "low criticality" |
| `report_activation_date` | Not in dim_report — not "recently launched" |

## 13. Deterministic Action Policy

Allowed recommended actions and their triggers:

In [ ]:
from src.analytics.report_diagnostics import ALLOWED_RECOMMENDED_ACTIONS, PROHIBITED_ACTIONS
from src.analytics.report_analytics_mart import PROHIBITED_MART_ACTIONS

print("Allowed recommended actions:")
for action in sorted(ALLOWED_RECOMMENDED_ACTIONS):
    print(f"  v {action}")

print("\nProhibited actions (never recommended):")
for action in sorted(PROHIBITED_ACTIONS | PROHIBITED_MART_ACTIONS):
    print(f"  x {action}")

# Verify no prohibited action appears in any output
outputs_to_check = {
    "diagnostics": diagnostics_df,
    "mart": mart_df,
}
all_prohibited = PROHIBITED_ACTIONS | PROHIBITED_MART_ACTIONS
for name, df in outputs_to_check.items():
    for col in ["recommended_diagnostic_action", "recommended_report_action", "recommended_model_action"]:
        if col in df.columns:
            actual = set(df[col].dropna())
            violations = actual & all_prohibited
            print(f"\n{name}.{col}: {len(actual)} unique actions, violations={violations or 'none'}")

## 14. Relationship to Later Sprints

| Sprint | Consumer | Input from Sprint 7 |
|---|---|---|
| Sprint 8 | GenAI narrative generation | `mart_report_analytics.csv` |
| Sprint 9 | Streamlit dashboard | `mart_report_analytics.csv` + segments + diagnostics |

Sprint 7 itself does **not** modify GenAI or Streamlit code. Those pipelines will read
the persisted outputs from `outputs/analytics/` as their starting point.

## 15. Limitations

1. **Business value may not be observable** — a report used by one critical decision-maker
   may have low view counts but high importance
2. **Criticality and cadence may be missing** — when not in `dim_report`, they default to
   "unknown" and do not imply low importance
3. **Forecasts remain uncertain** — especially for reports with volatile or sparse history
4. **Model diagnostics may show immature production evidence** — backtests may be limited
   for recently deployed models
5. **Privacy suppression limits small-group detail** — groups with < 5 users have suppressed
   distribution metrics; this is a feature, not a bug
6. **Concentrated or low-frequency usage may be appropriate** — a report used daily by
   3 executives is not a problem; it is a dependency risk requiring awareness
7. **Report retirement requires stakeholder review** — the pipeline never recommends retirement;
   humans make that decision
8. **Synthetic data may not represent production behaviour** — thresholds and distributions
   in this notebook reflect the synthetic training dataset